<a href="https://colab.research.google.com/github/goitstudent123/llm-gen/blob/main/dz_final_has.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# Chunk 1: Install dependencies
!pip -q install -U pandas openpyxl requests tqdm langchain langchain-openai
!pip -q check

In [12]:
# Chunk 2: Initial imports and OpenRouter setup
import os
import re
import json
import math
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from tqdm.auto import tqdm
from IPython.display import display, Markdown

from langchain_openai import ChatOpenAI


def read_colab_secret(name: str) -> Optional[str]:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

openrouter_api_key = (
    read_colab_secret("OPENROUTER_API_KEY")
    or os.getenv("OPENROUTER_API_KEY")
)

OPENROUTER_MODEL = (
    read_colab_secret("OPENROUTER_MODEL")
    or os.getenv("OPENROUTER_MODEL")
    or "openai/gpt-4.1-mini"
)

OPENROUTER_HTTP_REFERER = (
    read_colab_secret("OPENROUTER_HTTP_REFERER")
    or os.getenv("OPENROUTER_HTTP_REFERER")
    or "https://colab.research.google.com"
)

OPENROUTER_APP_TITLE = (
    read_colab_secret("OPENROUTER_APP_TITLE")
    or os.getenv("OPENROUTER_APP_TITLE")
    or "Colab Excel Enrichment"
)

llm = None

if openrouter_api_key:
    llm = ChatOpenAI(
        model=OPENROUTER_MODEL,
        api_key=openrouter_api_key,
        base_url=OPENROUTER_BASE_URL,
        temperature=0,
        default_headers={
            "HTTP-Referer": OPENROUTER_HTTP_REFERER,
            "X-OpenRouter-Title": OPENROUTER_APP_TITLE,
        },
    )
    print(f"OpenRouter model: {OPENROUTER_MODEL}")
else:
    print("OPENROUTER_API_KEY was not found. The deterministic planner will be used.")

OpenRouter model: openai/gpt-4.1-mini


In [13]:
# Chunk 3: Colab file upload and Google Sheets download helpers
def upload_excel_files() -> List[str]:
    try:
        from google.colab import files
        uploaded = files.upload()
        return list(uploaded.keys())
    except Exception as error:
        print(f"Upload is available only in Colab: {error}")
        return []


def download_result_file(file_path: str) -> None:
    try:
        from google.colab import files
        files.download(file_path)
    except Exception as error:
        print(f"Download is available only in Colab: {error}")


def extract_google_sheet_id(url: str) -> str:
    match = re.search(r"/spreadsheets/d/([a-zA-Z0-9-_]+)", url)
    if not match:
        raise ValueError("Could not extract Google Sheet ID from URL.")
    return match.group(1)


def extract_google_sheet_gid(url: str) -> str:
    match = re.search(r"gid=([0-9]+)", url)
    return match.group(1) if match else "0"


def download_google_sheet_as_xlsx(url: str, output_path: str) -> str:
    sheet_id = extract_google_sheet_id(url)
    gid = extract_google_sheet_gid(url)

    export_url = (
        f"https://docs.google.com/spreadsheets/d/{sheet_id}/export"
        f"?format=xlsx&gid={gid}"
    )

    response = requests.get(export_url, timeout=30)

    if response.status_code != 200:
        raise RuntimeError(
            f"Could not download Google Sheet. Status: {response.status_code}"
        )

    Path(output_path).write_bytes(response.content)
    return output_path

In [14]:
# Chunk 4: Create sample Excel files
def create_sample_excel_files() -> Tuple[str, str]:
    capitals_df = pd.DataFrame(
        [
            {
                "Capital_From": "Kyiv",
                "Country_From": "Ukraine",
                "Capital_To": "Paris",
                "Country_To": "France",
                "distance": None,
            },
            {
                "Capital_From": "London",
                "Country_From": "United Kingdom",
                "Capital_To": "Rome",
                "Country_To": "Italy",
                "distance": None,
            },
            {
                "Capital_From": "Paris",
                "Country_From": "France",
                "Capital_To": "Berlin",
                "Country_To": "Germany",
                "distance": None,
            },
            {
                "Capital_From": "Berlin",
                "Country_From": "Germany",
                "Capital_To": "Vienna",
                "Country_To": "Austria",
                "distance": None,
            },
            {
                "Capital_From": "Warsaw",
                "Country_From": "Poland",
                "Capital_To": "Kyiv",
                "Country_To": "Ukraine",
                "distance": None,
            },
        ]
    )

    mountains_df = pd.DataFrame(
        [
            {"Mountain": "Everest", "Country": "Nepal", "height": None},
            {"Mountain": "Mont Blanc", "Country": "France", "height": None},
            {"Mountain": "Denali", "Country": "USA", "height": None},
            {"Mountain": "Kilimanjaro", "Country": "Tanzania", "height": None},
        ]
    )

    capitals_path = "capitals.xlsx"
    mountains_path = "mountains.xlsx"

    capitals_df.to_excel(capitals_path, index=False)
    mountains_df.to_excel(mountains_path, index=False)

    return capitals_path, mountains_path


capitals_path, mountains_path = create_sample_excel_files()

print(capitals_path)
print(mountains_path)

capitals.xlsx
mountains.xlsx


In [15]:
# Chunk 5: Analyze task and build enrichment plan
def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)

    if not match:
        return None

    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def find_column_case_insensitive(columns: List[str], expected_name: str) -> Optional[str]:
    expected_lower = expected_name.lower()

    for column in columns:
        if column.lower() == expected_lower:
            return column

    return None


def deterministic_plan(df: pd.DataFrame, task_description: str) -> Dict[str, Any]:
    columns = list(df.columns)
    text = task_description.lower()

    distance_column = find_column_case_insensitive(columns, "distance")
    height_column = find_column_case_insensitive(columns, "height")

    if distance_column or "відстан" in text or "distance" in text:
        return {
            "task_type": "distance_between_places",
            "target_column": distance_column or "distance",
            "required_columns": [
                "Capital_From",
                "Country_From",
                "Capital_To",
                "Country_To",
            ],
            "from_place_columns": ["Capital_From", "Country_From"],
            "to_place_columns": ["Capital_To", "Country_To"],
            "place_context_suffix": "capital city",
            "value_unit": "km",
        }

    if height_column or "висот" in text or "height" in text:
        return {
            "task_type": "wikidata_quantity_property",
            "target_column": height_column or "height",
            "required_columns": [
                "Mountain",
                "Country",
            ],
            "entity_column": "Mountain",
            "context_columns": ["Country"],
            "entity_context_suffix": "mountain",
            "wikidata_property_id": "P2044",
            "value_unit": "m",
        }

    return {
        "task_type": "unsupported",
        "target_column": "",
        "required_columns": [],
        "value_unit": "",
    }


def normalize_enrichment_plan(
    parsed: Optional[Dict[str, Any]],
    fallback_plan: Dict[str, Any],
) -> Dict[str, Any]:
    if not parsed:
        return fallback_plan

    task_type = parsed.get("task_type")

    if task_type not in {
        "distance_between_places",
        "wikidata_quantity_property",
        "unsupported",
    }:
        return fallback_plan

    if task_type == "unsupported":
        return {
            "task_type": "unsupported",
            "target_column": "",
            "required_columns": [],
            "value_unit": "",
        }

    if not parsed.get("target_column"):
        return fallback_plan

    if not parsed.get("required_columns"):
        return fallback_plan

    if task_type == "distance_between_places":
        if not parsed.get("from_place_columns") or not parsed.get("to_place_columns"):
            return fallback_plan

        return {
            "task_type": "distance_between_places",
            "target_column": parsed["target_column"],
            "required_columns": parsed["required_columns"],
            "from_place_columns": parsed["from_place_columns"],
            "to_place_columns": parsed["to_place_columns"],
            "place_context_suffix": parsed.get("place_context_suffix", ""),
            "value_unit": parsed.get("value_unit", "km"),
        }

    if task_type == "wikidata_quantity_property":
        if not parsed.get("entity_column") or not parsed.get("wikidata_property_id"):
            return fallback_plan

        return {
            "task_type": "wikidata_quantity_property",
            "target_column": parsed["target_column"],
            "required_columns": parsed["required_columns"],
            "entity_column": parsed["entity_column"],
            "context_columns": parsed.get("context_columns", []),
            "entity_context_suffix": parsed.get("entity_context_suffix", ""),
            "wikidata_property_id": parsed["wikidata_property_id"],
            "value_unit": parsed.get("value_unit", ""),
        }

    return fallback_plan


def analyze_task(df: pd.DataFrame, task_description: str) -> Dict[str, Any]:
    fallback_plan = deterministic_plan(df, task_description)

    if llm is None:
        return fallback_plan

    sample_rows = df.head(5).fillna("").to_dict(orient="records")

    prompt = f"""
You analyze Excel enrichment tasks.

Return JSON only with this schema:
{{
  "task_type": "distance_between_places|wikidata_quantity_property|unsupported",
  "target_column": "column name to fill or create",
  "required_columns": ["column names needed from the file"],
  "from_place_columns": ["name column", "optional context column"],
  "to_place_columns": ["name column", "optional context column"],
  "place_context_suffix": "optional search hint, for example capital city",
  "entity_column": "entity name column",
  "context_columns": ["optional context columns"],
  "entity_context_suffix": "optional search hint, for example mountain",
  "wikidata_property_id": "Wikidata property id, for example P2044",
  "value_unit": "km|m|"
}}

Use "distance_between_places" when the task asks to calculate distance from coordinates.
Use "wikidata_quantity_property" when the task asks to fill a numeric value from Wikidata.

Task description:
{task_description}

Columns:
{list(df.columns)}

Sample rows:
{sample_rows}
""".strip()

    try:
        response = llm.invoke(prompt)
        parsed = extract_json_object(response.content)
        return normalize_enrichment_plan(parsed, fallback_plan)
    except Exception:
        return fallback_plan


def validate_required_columns(df: pd.DataFrame, required_columns: List[str]) -> None:
    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

In [16]:
# Chunk 6: Wikidata internet lookup helpers
WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"
WIKIDATA_SPARQL_URL = "https://query.wikidata.org/sparql"

HTTP_HEADERS = {
    "User-Agent": "colab-excel-enrichment/1.0 educational project"
}


def request_json(url: str, params: Dict[str, Any], timeout: int = 30) -> Dict[str, Any]:
    response = requests.get(
        url,
        params=params,
        headers=HTTP_HEADERS,
        timeout=timeout,
    )
    response.raise_for_status()
    return response.json()


def search_wikidata_entities(query: str, limit: int = 10) -> List[Dict[str, Any]]:
    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": "en",
        "search": query,
        "limit": limit,
    }

    data = request_json(WIKIDATA_API_URL, params)
    return data.get("search", [])


def get_wikidata_entities(qids: List[str]) -> Dict[str, Any]:
    if not qids:
        return {}

    params = {
        "action": "wbgetentities",
        "format": "json",
        "ids": "|".join(qids),
        "props": "claims|labels|descriptions",
        "languages": "en",
    }

    data = request_json(WIKIDATA_API_URL, params)
    return data.get("entities", {})


def sparql_escape(value: str) -> str:
    return value.replace("\\", "\\\\").replace('"', '\\"')


def query_wikidata_sparql(query: str, timeout: int = 30) -> Dict[str, Any]:
    params = {
        "query": query,
        "format": "json",
    }

    return request_json(WIKIDATA_SPARQL_URL, params, timeout=timeout)


def get_claim_values(entity: Dict[str, Any], property_id: str) -> List[Any]:
    claims = entity.get("claims", {}).get(property_id, [])
    values = []

    for claim in claims:
        main_snak = claim.get("mainsnak", {})
        data_value = main_snak.get("datavalue", {})

        if "value" in data_value:
            values.append(data_value["value"])

    return values


def get_claim_value(entity: Dict[str, Any], property_id: str) -> Optional[Any]:
    values = get_claim_values(entity, property_id)

    if not values:
        return None

    return values[0]


def extract_coordinates(entity: Dict[str, Any]) -> Optional[Tuple[float, float]]:
    values = get_claim_values(entity, "P625")

    for value in values:
        if isinstance(value, dict) and "latitude" in value and "longitude" in value:
            return float(value["latitude"]), float(value["longitude"])

    return None


def extract_quantity_amount(
    entity: Dict[str, Any],
    property_id: str,
) -> Optional[int]:
    values = get_claim_values(entity, property_id)

    for value in values:
        if isinstance(value, dict) and "amount" in value:
            return int(round(float(value["amount"])))

    return None


def entity_label(entity: Dict[str, Any]) -> str:
    labels = entity.get("labels", {})
    english_label = labels.get("en", {})

    return english_label.get("value", "")


def entity_description(entity: Dict[str, Any]) -> str:
    descriptions = entity.get("descriptions", {})
    english_description = descriptions.get("en", {})

    return english_description.get("value", "")


def entity_url(qid: str) -> str:
    return f"https://www.wikidata.org/wiki/{qid}"


def qid_from_uri(uri: str) -> str:
    return uri.rstrip("/").split("/")[-1]


def parse_wikidata_point(value: str) -> Optional[Tuple[float, float]]:
    match = re.match(r"Point\(([-\d.]+) ([-\d.]+)\)", value)

    if not match:
        return None

    longitude = float(match.group(1))
    latitude = float(match.group(2))

    return latitude, longitude

In [17]:
# Chunk 7: Generic enrichment lookup logic
def clean_text(value: Any) -> str:
    if pd.isna(value):
        return ""

    return str(value).strip()


def build_search_text(*parts: Any) -> str:
    return " ".join(
        part
        for part in (clean_text(value) for value in parts)
        if part
    )


def normalize_for_match(value: Any) -> str:
    text = clean_text(value).lower()
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def split_context_values(context: str) -> List[str]:
    raw_parts = re.split(r"[/,;|]+", clean_text(context))

    return [
        part.strip()
        for part in raw_parts
        if part.strip()
    ]


def unique_texts(values: List[str]) -> List[str]:
    result = []

    for value in values:
        if value and value not in result:
            result.append(value)

    return result


def haversine_km(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float,
) -> float:
    earth_radius_km = 6371.0088

    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    delta_lat = math.radians(lat2 - lat1)
    delta_lon = math.radians(lon2 - lon1)

    value = (
        math.sin(delta_lat / 2) ** 2
        + math.cos(lat1_rad)
        * math.cos(lat2_rad)
        * math.sin(delta_lon / 2) ** 2
    )

    central_angle = 2 * math.atan2(math.sqrt(value), math.sqrt(1 - value))
    return earth_radius_km * central_angle


def context_score(text: str, context: str) -> int:
    normalized_text = normalize_for_match(text)
    score = 0

    for part in split_context_values(context):
        normalized_part = normalize_for_match(part)

        if normalized_part and normalized_part in normalized_text:
            score += 1

    return score


def resolve_place_coordinates_by_sparql(
    place_name: str,
    context: str = "",
) -> Optional[Dict[str, Any]]:
    escaped_place_name = sparql_escape(clean_text(place_name))

    query = f"""
SELECT ?item ?itemLabel ?coord ?countryLabel WHERE {{
  ?item rdfs:label "{escaped_place_name}"@en.
  ?item wdt:P625 ?coord.

  OPTIONAL {{
    ?item wdt:P17 ?country.
    ?country rdfs:label ?countryLabel.
    FILTER(LANG(?countryLabel) = "en")
  }}

  SERVICE wikibase:label {{
    bd:serviceParam wikibase:language "en".
    ?item rdfs:label ?itemLabel.
  }}
}}
LIMIT 25
""".strip()

    data = query_wikidata_sparql(query)
    bindings = data.get("results", {}).get("bindings", [])

    if not bindings:
        return None

    candidates = []

    for binding in bindings:
        item_uri = binding.get("item", {}).get("value", "")
        item_label = binding.get("itemLabel", {}).get("value", "")
        country_label = binding.get("countryLabel", {}).get("value", "")
        coord_value = binding.get("coord", {}).get("value", "")

        coordinates = parse_wikidata_point(coord_value)

        if not item_uri or not coordinates:
            continue

        qid = qid_from_uri(item_uri)
        combined_text = build_search_text(item_label, country_label)
        score = context_score(combined_text, context)

        candidates.append(
            {
                "ok": True,
                "coordinates": coordinates,
                "source": entity_url(qid),
                "status": "wikidata",
                "score": score,
            }
        )

    if not candidates:
        return None

    candidates.sort(key=lambda item: item["score"], reverse=True)

    best_candidate = candidates[0]
    best_candidate.pop("score", None)

    return best_candidate


def resolve_place_coordinates_by_api_search(
    place_name: str,
    context: str = "",
    context_suffix: str = "",
) -> Optional[Dict[str, Any]]:
    search_queries = unique_texts(
        [
            build_search_text(place_name, context, context_suffix),
            build_search_text(place_name, context),
            build_search_text(place_name, context_suffix),
            build_search_text(place_name),
        ]
    )

    candidates = []

    for query in search_queries:
        try:
            search_results = search_wikidata_entities(query, limit=20)
            qids = [item["id"] for item in search_results if item.get("id")]
            entities = get_wikidata_entities(qids)

            for qid in qids:
                entity = entities.get(qid, {})
                coordinates = extract_coordinates(entity)

                if not coordinates:
                    continue

                combined_text = build_search_text(
                    entity_label(entity),
                    entity_description(entity),
                )

                score = context_score(combined_text, context)

                candidates.append(
                    {
                        "ok": True,
                        "coordinates": coordinates,
                        "source": entity_url(qid),
                        "status": "wikidata",
                        "score": score,
                    }
                )
        except Exception:
            continue

    if not candidates:
        return None

    candidates.sort(key=lambda item: item["score"], reverse=True)

    best_candidate = candidates[0]
    best_candidate.pop("score", None)

    return best_candidate


def resolve_place_coordinates(
    place_name: str,
    context: str = "",
    context_suffix: str = "",
) -> Dict[str, Any]:
    try:
        result = resolve_place_coordinates_by_sparql(place_name, context)

        if result:
            return result
    except Exception:
        pass

    result = resolve_place_coordinates_by_api_search(
        place_name=place_name,
        context=context,
        context_suffix=context_suffix,
    )

    if result:
        return result

    return {
        "ok": False,
        "coordinates": None,
        "source": "",
        "status": "not_found",
    }


def resolve_wikidata_quantity_by_sparql(
    entity_name: str,
    property_id: str,
    context: str = "",
) -> Optional[Dict[str, Any]]:
    escaped_entity_name = sparql_escape(clean_text(entity_name))

    query = f"""
SELECT ?item ?itemLabel ?value ?countryLabel WHERE {{
  ?item rdfs:label "{escaped_entity_name}"@en.
  ?item wdt:{property_id} ?value.

  OPTIONAL {{
    ?item wdt:P17 ?country.
    ?country rdfs:label ?countryLabel.
    FILTER(LANG(?countryLabel) = "en")
  }}

  SERVICE wikibase:label {{
    bd:serviceParam wikibase:language "en".
    ?item rdfs:label ?itemLabel.
  }}
}}
LIMIT 25
""".strip()

    data = query_wikidata_sparql(query)
    bindings = data.get("results", {}).get("bindings", [])

    if not bindings:
        return None

    candidates = []

    for binding in bindings:
        item_uri = binding.get("item", {}).get("value", "")
        item_label = binding.get("itemLabel", {}).get("value", "")
        country_label = binding.get("countryLabel", {}).get("value", "")
        raw_value = binding.get("value", {}).get("value")

        if not item_uri or raw_value is None:
            continue

        try:
            numeric_value = int(round(float(raw_value)))
        except ValueError:
            continue

        qid = qid_from_uri(item_uri)
        combined_text = build_search_text(item_label, country_label)
        score = context_score(combined_text, context)

        candidates.append(
            {
                "ok": True,
                "value": numeric_value,
                "source": entity_url(qid),
                "status": "wikidata",
                "score": score,
            }
        )

    if not candidates:
        return None

    candidates.sort(key=lambda item: item["score"], reverse=True)

    best_candidate = candidates[0]
    best_candidate.pop("score", None)

    return best_candidate


def resolve_wikidata_quantity_by_api_search(
    entity_name: str,
    context: str,
    property_id: str,
    context_suffix: str = "",
) -> Optional[Dict[str, Any]]:
    search_queries = unique_texts(
        [
            build_search_text(entity_name, context, context_suffix),
            build_search_text(entity_name, context),
            build_search_text(entity_name, context_suffix),
            build_search_text(entity_name),
        ]
    )

    candidates = []

    for query in search_queries:
        try:
            search_results = search_wikidata_entities(query, limit=20)
            qids = [item["id"] for item in search_results if item.get("id")]
            entities = get_wikidata_entities(qids)

            for qid in qids:
                entity = entities.get(qid, {})
                value = extract_quantity_amount(entity, property_id)

                if value is None:
                    continue

                combined_text = build_search_text(
                    entity_label(entity),
                    entity_description(entity),
                )

                score = context_score(combined_text, context)

                candidates.append(
                    {
                        "ok": True,
                        "value": value,
                        "source": entity_url(qid),
                        "status": "wikidata",
                        "score": score,
                    }
                )
        except Exception:
            continue

    if not candidates:
        return None

    candidates.sort(key=lambda item: item["score"], reverse=True)

    best_candidate = candidates[0]
    best_candidate.pop("score", None)

    return best_candidate


def resolve_wikidata_quantity_property(
    entity_name: str,
    context: str,
    property_id: str,
    context_suffix: str = "",
) -> Dict[str, Any]:
    try:
        result = resolve_wikidata_quantity_by_sparql(
            entity_name=entity_name,
            property_id=property_id,
            context=context,
        )

        if result:
            return result
    except Exception:
        pass

    result = resolve_wikidata_quantity_by_api_search(
        entity_name=entity_name,
        context=context,
        property_id=property_id,
        context_suffix=context_suffix,
    )

    if result:
        return result

    return {
        "ok": False,
        "value": None,
        "source": "",
        "status": "not_found",
    }

In [18]:
# Chunk 8: Excel processing and saving functions
def is_empty_value(value: Any) -> bool:
    if pd.isna(value):
        return True

    return str(value).strip() == ""


def format_excel_file(file_path: str) -> None:
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    workbook = load_workbook(file_path)
    worksheet = workbook.active

    header_fill = PatternFill("solid", fgColor="EAF2F8")

    for cell in worksheet[1]:
        cell.font = Font(bold=True)
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center")

    worksheet.freeze_panes = "A2"

    for column_cells in worksheet.columns:
        max_length = max(
            len(str(cell.value)) if cell.value is not None else 0
            for cell in column_cells
        )
        column_letter = get_column_letter(column_cells[0].column)
        worksheet.column_dimensions[column_letter].width = min(max(max_length + 2, 12), 45)

    workbook.save(file_path)


def build_context_from_columns(row: pd.Series, columns: List[str]) -> str:
    return build_search_text(*(row[column] for column in columns))


def build_place_lookup_args(
    row: pd.Series,
    place_columns: List[str],
) -> Tuple[str, str]:
    place_name = clean_text(row[place_columns[0]])
    context = build_context_from_columns(row, place_columns[1:])
    return place_name, context


def enrich_place_distances(
    df: pd.DataFrame,
    target_column: str,
    from_place_columns: List[str],
    to_place_columns: List[str],
    required_columns: List[str],
    place_context_suffix: str,
    overwrite: bool,
    include_debug_columns: bool,
    max_workers: int,
) -> pd.DataFrame:
    validate_required_columns(df, required_columns)

    if target_column not in df.columns:
        df[target_column] = None

    source_column = f"{target_column}_source"
    status_column = f"{target_column}_status"

    if include_debug_columns:
        df[source_column] = ""
        df[status_column] = ""

    unique_places = set()

    for _, row in df.iterrows():
        unique_places.add(build_place_lookup_args(row, from_place_columns))
        unique_places.add(build_place_lookup_args(row, to_place_columns))

    coordinate_results = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {
            executor.submit(
                resolve_place_coordinates,
                place_name,
                context,
                place_context_suffix,
            ): (place_name, context)
            for place_name, context in unique_places
        }

        for future in tqdm(as_completed(future_map), total=len(future_map), desc="Resolving places"):
            key = future_map[future]
            try:
                coordinate_results[key] = future.result()
            except Exception as error:
                coordinate_results[key] = {
                    "ok": False,
                    "coordinates": None,
                    "source": "",
                    "status": f"error: {error}",
                }

    for index, row in df.iterrows():
        current_value = row.get(target_column)

        if not overwrite and not is_empty_value(current_value):
            continue

        from_key = build_place_lookup_args(row, from_place_columns)
        to_key = build_place_lookup_args(row, to_place_columns)

        from_result = coordinate_results.get(from_key)
        to_result = coordinate_results.get(to_key)

        if not from_result or not to_result or not from_result["ok"] or not to_result["ok"]:
            df.at[index, target_column] = None

            if include_debug_columns:
                df.at[index, source_column] = ""
                df.at[index, status_column] = "not_found"

            continue

        lat1, lon1 = from_result["coordinates"]
        lat2, lon2 = to_result["coordinates"]

        distance_km = round(haversine_km(lat1, lon1, lat2, lon2))
        df.at[index, target_column] = distance_km

        if include_debug_columns:
            df.at[index, source_column] = f"{from_result['source']} | {to_result['source']}"
            df.at[index, status_column] = "wikidata"

    return df


def enrich_wikidata_quantity_property(
    df: pd.DataFrame,
    target_column: str,
    entity_column: str,
    context_columns: List[str],
    required_columns: List[str],
    wikidata_property_id: str,
    entity_context_suffix: str,
    overwrite: bool,
    include_debug_columns: bool,
    max_workers: int,
) -> pd.DataFrame:
    validate_required_columns(df, required_columns)

    if target_column not in df.columns:
        df[target_column] = None

    source_column = f"{target_column}_source"
    status_column = f"{target_column}_status"

    if include_debug_columns:
        df[source_column] = ""
        df[status_column] = ""

    unique_entities = {
        (
            clean_text(row[entity_column]),
            build_context_from_columns(row, context_columns),
        )
        for _, row in df.iterrows()
    }

    quantity_results = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {
            executor.submit(
                resolve_wikidata_quantity_property,
                entity_name,
                context,
                wikidata_property_id,
                entity_context_suffix,
            ): (entity_name, context)
            for entity_name, context in unique_entities
        }

        for future in tqdm(as_completed(future_map), total=len(future_map), desc="Resolving Wikidata values"):
            key = future_map[future]
            try:
                quantity_results[key] = future.result()
            except Exception as error:
                quantity_results[key] = {
                    "ok": False,
                    "value": None,
                    "source": "",
                    "status": f"error: {error}",
                }

    for index, row in df.iterrows():
        current_value = row.get(target_column)

        if not overwrite and not is_empty_value(current_value):
            continue

        key = (
            clean_text(row[entity_column]),
            build_context_from_columns(row, context_columns),
        )
        result = quantity_results.get(key)

        if not result or not result["ok"]:
            df.at[index, target_column] = None

            if include_debug_columns:
                df.at[index, source_column] = ""
                df.at[index, status_column] = "not_found"

            continue

        df.at[index, target_column] = result["value"]

        if include_debug_columns:
            df.at[index, source_column] = result["source"]
            df.at[index, status_column] = result["status"]

    return df


def process_excel(
    file_path: str,
    task_description: str,
    output_path: Optional[str] = None,
    overwrite: bool = True,
    include_debug_columns: bool = True,
    max_workers: int = 4,
) -> str:
    df = pd.read_excel(file_path)
    plan = analyze_task(df, task_description)

    print("Enrichment plan:")
    print(json.dumps(plan, ensure_ascii=False, indent=2))

    task_type = plan["task_type"]
    target_column = plan["target_column"]

    if task_type == "unsupported":
        raise ValueError("Unsupported enrichment task.")

    if task_type == "distance_between_places":
        enriched_df = enrich_place_distances(
            df=df,
            target_column=target_column,
            from_place_columns=plan["from_place_columns"],
            to_place_columns=plan["to_place_columns"],
            required_columns=plan["required_columns"],
            place_context_suffix=plan.get("place_context_suffix", ""),
            overwrite=overwrite,
            include_debug_columns=include_debug_columns,
            max_workers=max_workers,
        )
    elif task_type == "wikidata_quantity_property":
        enriched_df = enrich_wikidata_quantity_property(
            df=df,
            target_column=target_column,
            entity_column=plan["entity_column"],
            context_columns=plan.get("context_columns", []),
            required_columns=plan["required_columns"],
            wikidata_property_id=plan["wikidata_property_id"],
            entity_context_suffix=plan.get("entity_context_suffix", ""),
            overwrite=overwrite,
            include_debug_columns=include_debug_columns,
            max_workers=max_workers,
        )
    else:
        raise ValueError(f"Unsupported task type: {task_type}")

    if output_path is None:
        input_path = Path(file_path)
        output_path = str(input_path.with_name(f"{input_path.stem}_enriched.xlsx"))

    enriched_df.to_excel(output_path, index=False)
    format_excel_file(output_path)

    print(f"Saved: {output_path}")
    return output_path

In [19]:
# Chunk 9: Download correct input files and run enrichment
capitals_url = "https://docs.google.com/spreadsheets/d/1Bv_WFNzGG4ZL7-QQf8dSrnSEwRvUnhrh/edit?gid=28170114#gid=28170114"
mountains_url = "https://docs.google.com/spreadsheets/d/1UutRbt0yvinVpz3cQdAcRSlqPFmyVEXx/edit?gid=1262408660#gid=1262408660"

download_google_sheet_as_xlsx(capitals_url, "capitals.xlsx")
download_google_sheet_as_xlsx(mountains_url, "mountains.xlsx")

capitals_output = process_excel(
    file_path="capitals.xlsx",
    task_description="знайди пряму відстань між столицями в км для колонки distance",
    output_path="capitals_enriched.xlsx",
    include_debug_columns=False,
)

mountains_output = process_excel(
    file_path="mountains.xlsx",
    task_description="додай висоту гір у метрах до колонки height",
    output_path="mountains_enriched.xlsx",
    include_debug_columns=False,
)

print(capitals_output)
print(mountains_output)

Enrichment plan:
{
  "task_type": "distance_between_places",
  "target_column": "distance",
  "required_columns": [
    "Capital_From",
    "Country_From",
    "Capital_To",
    "Country_To"
  ],
  "from_place_columns": [
    "Capital_From",
    "Country_From"
  ],
  "to_place_columns": [
    "Capital_To",
    "Country_To"
  ],
  "place_context_suffix": "capital city",
  "value_unit": "km"
}


Resolving places:   0%|          | 0/14 [00:00<?, ?it/s]

Saved: capitals_enriched.xlsx
Enrichment plan:
{
  "task_type": "wikidata_quantity_property",
  "target_column": "height",
  "required_columns": [
    "Mountain"
  ],
  "entity_column": "Mountain",
  "context_columns": [
    "Country"
  ],
  "entity_context_suffix": "mountain",
  "wikidata_property_id": "P2044",
  "value_unit": "m"
}


Resolving Wikidata values:   0%|          | 0/10 [00:00<?, ?it/s]

Saved: mountains_enriched.xlsx
capitals_enriched.xlsx
mountains_enriched.xlsx


In [20]:
# Chunk 10: Preview and validate enriched Excel data
capitals_result = pd.read_excel("capitals_enriched.xlsx")
mountains_result = pd.read_excel("mountains_enriched.xlsx")

display(Markdown("## Capitals enriched"))
display(capitals_result)

display(Markdown("## Mountains enriched"))
display(mountains_result)

assert "distance_source" not in capitals_result.columns
assert "distance_status" not in capitals_result.columns
assert "height_source" not in mountains_result.columns
assert "height_status" not in mountains_result.columns

assert "distance" in capitals_result.columns
assert "height" in mountains_result.columns

assert capitals_result["distance"].notna().all()
assert mountains_result["height"].notna().all()

print("Validation passed: original structure preserved and target columns filled.")

## Capitals enriched

,Capital_From,Country_From,Capital_To,Country_To,distance
0,Kyiv,Ukraine,Paris,France,9265
1,London,United Kingdom,Rome,Italy,336
2,Paris,France,Berlin,Germany,7982
3,Berlin,Germany,Vienna,Austria,524
4,Warsaw,Poland,Kyiv,Ukraine,863
5,Rome,Italy,Madrid,Spain,1045
6,Madrid,Spain,Lisbon,Portugal,513
7,Vienna,Austria,Budapest,Hungary,214
8,Stockholm,Sweden,Oslo,Norway,416
9,Athens,Greece,Istanbul,Turkey,564


## Mountains enriched

,Mountain,Country,height
0,Mount Everest,Nepal/China,8849
1,K2,Pakistan/China,8611
2,Kangchenjunga,Nepal/India,8586
3,Lhotse,Nepal/China,8516
4,Makalu,Nepal/China,8485
5,Cho Oyu,Nepal/China,8188
6,Dhaulagiri,Nepal,8167
7,Manaslu,Nepal,8163
8,Nanga Parbat,Pakistan,8126
9,Annapurna,Nepal,7756


Validation passed: original structure preserved and target columns filled.


In [23]:
# Chunk 11: Download enriched files
download_result_file("capitals_enriched.xlsx")
download_result_file("mountains_enriched.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>